# Visualization  

In [ ]:
import torch
from gsm8k_dataset import GSMDataset, get_examples, split_solution_into_subsentences, calculate_score_M, classify_gsm8k_solution_segments, generate_prompt


In [ ]:
train_data = torch.load('gsm8k_train_data.pth')
test_data = torch.load('gsm8k_test_data.pth')

In [ ]:
import json
# load the train_set_labels from a file
with open('train_set_labels.json', 'r') as f:
    train_set_labels_loaded = json.load(f)
with open('test_set_labels.json', 'r') as f:
    test_set_labels_loaded = json.load(f)

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


label_to_num = {"AS": 1, "ES": 0}
metrics = list(train_set_labels_loaded.keys())
num_samples = len(train_set_labels_loaded[metrics[0]])

# Limit maximum number of sub-sentences shown
MAX_SEGMENTS = 10

# Compute padded matrices with truncated sub-sentences
data_matrices = {}
for metric in metrics:
    matrix = []
    for sample in train_set_labels_loaded[metric]:
        row = [label_to_num[label] for label in sample[:MAX_SEGMENTS]]
        # Pad if less than max segments
        row += [-1] * (MAX_SEGMENTS - len(row))
        matrix.append(row)
    data_matrices[metric] = np.array(matrix)

# Plot
fig, axes = plt.subplots(1, len(metrics), figsize=(16, 5), sharey=True)

for idx, metric in enumerate(metrics):
    ax = axes[idx]
    sns.heatmap(
        data_matrices[metric],
        ax=ax,
        cbar=False,
        cmap=sns.color_palette(["orange", "blue", "lightgray"]),
        vmin=-1, vmax=1,
        linewidths=0.5,
        xticklabels=[f"S{i+1}" for i in range(MAX_SEGMENTS)],
        yticklabels=[f"Sample {i+1}" for i in range(num_samples)]
    )
    ax.set_title(metric.capitalize())
    ax.set_xlabel("Sub-sentences")
    if idx == 0:
        ax.set_ylabel("Samples")

plt.tight_layout()
plt.suptitle(f"AS/ES Label Distribution (Only First {MAX_SEGMENTS} Sub-sentences)", y=1.08)
plt.show()

In [ ]:
i = 0
print(
train_set_labels_loaded["bleu"][i],
train_set_labels_loaded["entropy"][i],
train_set_labels_loaded["loss"][i],
train_set_labels_loaded["location"][i]
)

In [ ]:
type(train_data) == GSMDataset

In [ ]:
train_data.__getitemInTxt__(i)

In [ ]:
i=1
question, answer, grade = train_data.__getitemInTxt__(i)
transcript = train_data.__getTranscript__(i)
types = train_data.__getTypes__(i)
labels = train_set_labels_loaded[i]
input_txt, output_txt = generate_prompt(question=question, answer=answer, grade=grade, labels=labels, transcript=transcript)
print(input_txt, output_txt)


In [ ]:
import random

def generate_skill_labels_by_type(
    question_types=None,
    distribution=("weak", "normal", "strong"),
    bias_toward_weak=True
):
    """
    Generate skill labels (weak, normal, strong) for each question type.

    Args:
        question_types (list): A list of question types (e.g., ["addition", "subtraction"]).
        distribution (tuple): Skill levels to choose from.
        bias_toward_weak (bool): If True, increases chance of assigning 'weak'.

    Returns:
        dict: A mapping of question type to skill label.
    """

    if bias_toward_weak:
        weights = [0.7, 0.3, 0.0]  # More chance of 'weak'
    else:
        weights = [1 / len(distribution)] * len(distribution)


    result = {
        qt: random.choices(distribution, weights=weights, k=1)[0]
        for qt in question_types
    }

    all_types = ["ADD","MUL","SUB","DIV"]
    for type in all_types:
        if not type in result.keys():
            result[type] =  "strong"


    return result

In [ ]:
generate_skill_labels_by_type(question_types=["ADD","SUB"])

In [ ]:
# Default (balanced)
print(generate_skill_labels_by_type())

# Biased toward generating more 'weak' labels
print(generate_skill_labels_by_type(bias_toward_weak=True))

# Custom question types
custom_qtypes = ["fractions", "geometry", "money"]
print(generate_skill_labels_by_type(question_types=custom_qtypes))

## calculate BLEU

In [ ]:
import nltk
from evaluation_matrix import compute_bleu_one_reference_n_multiple_candidates
bleu_scores_sub_sentences = []
for idx in range(0, ):
    question, answer, grade = train_data.__getitemInTxt__(idx)
   
    sub_sentences = split_solution_into_subsentences(answer)

    for j, sentence in enumerate(sub_sentences):
        # print(f"  {j+1}: {sentence}")
        # calcylate BlEU by using corpus_bleu
        scores = compute_bleu_one_reference_n_multiple_candidates(
            reference=question, candidates=sub_sentences)
        # score_M = calculate_score_M(sentence, question, method="bleu")
        bleu_scores_sub_sentences.append(scores)
        # print(f"Score M: {score_M}")

In [ ]:
bleu_scores_sub_sentences[0]

## Calculate Location (interleaving)

In [ ]:
interleaving_sub_sentences = []
for idx in range(len(train_data)):
    question, answer, grade = train_data.__getitemInTxt__(idx)
   
    sub_sentences = split_solution_into_subsentences(answer)

    scores = []
    for j, sentence in enumerate(sub_sentences):
        # print(f"  {j+1}: {sentence}")
        if j % 2 == 0:
            scores.append(0)
        else:
            scores.append(1)
        # print(f"Score M: {score_M}")

    interleaving_sub_sentences.append(scores)

In [ ]:
interleaving_sub_sentences[0:10]

In [ ]:
for i, problem in enumerate(train_data):
        
    if i >= 5:  # Limit to first 5 problems for demonstration
        break
    
    problem.
    print(f"Problem {i+1}: {problem['que']}")
    print("Solution:", problem['ans'])
    
    # Example of splitting the solution into subsentences
    subsentences = split_solution_into_subsentences(problem['solution'])
    print("Subsentences:")
    for j, sentence in enumerate(subsentences):
        print(f"  {j+1}: {sentence}")

    # # Example of calculating score M
    # score_M = calculate_score_M(problem['solution'])
    # print(f"Score M: {score_M}")

    # # Example of classifying segments of the solution
    # for method in ['interleaving', 'bleu']:
    #     beta_val = 0.5 if method == 'heuristic' else None
    #     print(f"\nClassifying segments using {method} method with beta={beta_val}:")


# classified_solution = classify_gsm8k_solution_segments(problem, segmentation_method=method, beta=beta_val)
#             for segment in classified_solution:
#                 print(f"  [{segment['type']}] ({segment['reason']}): {segment['segment_text']}")
#             print("\n" + "="*50 + "\n")


# using Flan T5 to calculate entropy and loss

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize
nltk.download("punkt")



In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
model_name = "google/flan-t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

In [ ]:
import numpy as np

def segment_cot(cot_text):
    solution_core = cot_text.split('####')[0].strip()
    return sent_tokenize(solution_core)

def calculate_entropy(model, tokenizer, query, response_sub_sentence):
    """Calculate entropy for a sub-sentence in the response given the query."""
    # input_text = f"{query} {response_sub_sentence}"
    # inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
    # with torch.no_grad():
    #     outputs = model(**inputs)

    # Prepare encoder inputs
    encoder_inputs = tokenizer(
        query, 
        return_tensors="pt", 
        truncation=True, 
        max_length=512
    )
    # Prepare decoder inputs (teacher forcing)
    decoder_inputs = tokenizer(
        response_sub_sentence,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        # Forward pass with encoder and decoder inputs
        outputs = model(
            input_ids=encoder_inputs.input_ids,
            attention_mask=encoder_inputs.attention_mask,
            decoder_input_ids=decoder_inputs.input_ids,
            decoder_attention_mask=decoder_inputs.attention_mask,
            return_dict=True
        )    

    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-10), dim=-1).mean().item()
    return entropy

def calculate_metrics(model, tokenizer, query, sub_sentence):
    """Returns both loss and entropy for a sub-sentence given the query."""
    # Tokenize inputs
    encoder_inputs = tokenizer(query, return_tensors="pt", truncation=True, max_length=512)
    decoder_inputs = tokenizer(sub_sentence, return_tensors="pt", truncation=True, max_length=512)
    
    # Prepare labels (shifted for teacher forcing)
    labels = decoder_inputs.input_ids.clone()
    labels[labels == tokenizer.pad_token_id] = -100  # Ignore padding in loss
    
    with torch.no_grad():
        outputs = model(
            input_ids=encoder_inputs.input_ids,
            attention_mask=encoder_inputs.attention_mask,
            decoder_input_ids=decoder_inputs.input_ids,
            decoder_attention_mask=decoder_inputs.attention_mask,
            labels=labels,
            return_dict=True
        )
        
        # Get loss (cross-entropy)
        loss = outputs.loss.item()
        
        # Calculate entropy from logits
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-10), dim=-1).mean().item()
    
    return {"loss": loss, "entropy": entropy}

def classify_as_es(model, tokenizer, query, cot_text, beta=1.0):
    """Classify sub-sentences in CoT into AS or ES using entropy-oriented segmentation."""
    sub_sentences = segment_cot(cot_text)
    entropy_n_loss = []
    for sub_sentence in sub_sentences:
        scores = calculate_metrics(model, tokenizer, query, sub_sentence)
        entropy_n_loss.append(scores)
    
    # avg_entropy = np.mean(entropies)
    as_segments = []
    es_segments = []
    
    # for sub_sentence, entropy in zip(sub_sentences, entropies):
    #     if entropy > beta * avg_entropy:
    #         as_segments.append(sub_sentence)
    #     else:
    #         es_segments.append(sub_sentence)
    
    return as_segments, es_segments

In [ ]:
from tqdm import tqdm 

start = 0
end = len(train_data)
# Classify each example in the dataset
all_entropy_n_loss = []
with torch.no_grad():
    for idx in tqdm(range(start, end, 1), desc="Processing data"): # Process first 10 examples for demonstration
        question, answer, grade = train_data.__getitemInTxt__(idx)
        query = question
        cot_text = answer  # Assuming the answer contains CoT
        sub_sentences = segment_cot(cot_text)
        entropy_n_loss = []
        # print(f"Processing example {idx + 1}/{len(train_data)}: {query}")
        for sub_sentence in sub_sentences:
            entropy_n_loss.append(calculate_metrics(model, tokenizer, query, sub_sentence))
        all_entropy_n_loss.append(entropy_n_loss)
    # as_segments, es_segments = classify_as_es(model, tokenizer, query, cot_text)
        
    # classified_example = {
    #     "question": query,
    #     "answer": cot_text,
    #     "AS": as_segments,
    #     "ES": es_segments
    # }
    # classified_data.append(classified_example)
    # print(f"Question: {query}")
    # print(f"AS Segments: {as_segments}")
    # print(f"ES Segments: {es_segments}")
    # print("-" * 50)

In [ ]:
#export all_entropy_n_loss
torch.save(all_entropy_n_loss, 'gsm8k_train_set_entropy_n_loss.pth')

In [ ]:
all_entropy_n_loss = torch.load('gsm8k_train_set_entropy_n_loss.pth')

In [ ]:
all_entropy_n_loss[0][0]["loss"]

for idx in range(0, 5):
    for idx_sentence in range(0, len(all_entropy_n_loss[idx])):
        sub_sentence = all_entropy_n_loss[idx][idx_sentence]
        print(f"answer {idx} - sentence {idx_sentence}: Loss: {sub_sentence['loss']}, Entropy: {sub_sentence['entropy']}")

In [ ]:
len(all_entropy_n_loss)

____

In [ ]:
grade_count = {}
for idx in range(len(train_data)):
    grade = train_data.__getGrade__(idx)  # Note the idx parameter
    if grade not in grade_count:
        grade_count[grade] = 0
    grade_count[grade] += 1

In [ ]:
grade_count

In [ ]:
import re
instances = {}
for idx in range(len(train_data)):
    grade = train_data.__getGrade__(idx)  # Note the idx parameter
    if grade == 1:
        question, answer, grade = train_data.__getitemInTxt__(idx)  # Note the idx parameter
        if re.search(r"\*", answer):
            instances[idx] = (question, answer, grade)

In [ ]:
print(len(instances))
print("Instances with grade 9:")
# loop instances with start and end
keys = sorted(instances.keys())
start = 0

end = start + 5
for i in range(start, len(keys), end):
    current_keys = keys[i:i+10]
    print(f"Processing idx {current_keys}:")
    
    for key in current_keys:
        question, answer, grade = instances[key]
        print(f"Index: {key}, Grade: {grade}")
        print(f"Question: {question}")
        print(f"Answer: {answer}")
        print("-" * 40)
    # question, answer, grade = instances[idx]
    # print(f"Index: {idx}, Grade: {grade}")
    # print(f"Question: {question}")
    # print(f"Answer: {answer}")
    # print("-" * 40)

In [ ]:
train_data.__setGrade__(145, 4)  # Set the grade of the first instance to 9

In [ ]:
train_data.__getitemInTxt__(145)  # Verify the grade has been set to 9

# Read GSM8K

In [ ]:
# read_data.py
import sys
from pathlib import Path

In [ ]:
from gsm8k_dataset import GSMDataset, get_examples, split_solution_into_subsentences
from transformers import GPT2Tokenizer, GPT2LMHeadModel


tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
train_examples = get_examples("train")
test_examples = get_examples("test")
train_dset = GSMDataset(tokenizer, train_examples)
test_dset = GSMDataset(tokenizer, test_examples)

In [ ]:
# set the grade to new loaded dataset
import torch
train_set = torch.load('gsm8k_train_data.pth')
test_set = torch.load('gsm8k_test_data.pth')

for idx, row in enumerate(train_set):
    question, answer, grade =  train_set.__getitemInTxt__(idx)
    train_dset.__setGrade__(idx, grade)

for idx, row in enumerate(test_set):
    question, answer, grade =  test_set.__getitemInTxt__(idx)
    test_dset.__setGrade__(idx, grade)

In [ ]:
q, a, g = test_dset.__getitemInTxt__(1)
split_solution_into_subsentences(a)

In [ ]:
# export test_dset
torch.save(train_dset, 'gsm8k_train_data.pth')
torch.save(test_dset, 'gsm8k_test_data.pth')

In [ ]:
import re
symbols_to_types = {
    "+":"ADD",
    "-":"SUB",
    "*":"MUL",
    "/":"DIV"
}

results = []
for idx, instance in enumerate(train_dset):
    types = []
    if idx >= 5:  # Limit to first 5 instances for demonstration
        break

    question, answer, grade = train_dset.__getitemInTxt__(idx)
    print(f"Index: {idx}, answer: {answer}")
    # captures only the text into << >> from the answer
    matches = re.findall(r"<<(.*?)>>", answer)
    for match in matches:
        # replace the math symbols with their corresponding types
        match = match.strip()
        for symbol, type_ in symbols_to_types.items():
            if symbol in match:
                if type_ not in types:
                    types.append(type_)
    print(matches)
    print(f"Types: {types}")
    print("-" * 40)

In [ ]:
lst = [5, 4, 3, 5, 5, 4, 5, 6, 3]


set_grade(lst,create_index_list(start, end-1))
print(start, end)
test_dset.__getitemInTxt__(end-2)  # Get the first item in text format

In [ ]:
result = []
for i in range(0, end-1):
    a, b, c = test_dset.__getitemInTxt__(i)
    if c == None:
        result.append(f"{i} No grade")
result

In [ ]:
def set_grade(grade_list, idx_list):
    for j, idx in enumerate(idx_list):
        test_dset.__setGrade__(idx, grade_list[j])


In [ ]:
for i in range(0, len(train_data)):
    q,a,g = train_data.__getitemInTxt__(i)
    q2, qa, qg = train_dset.__getitemInTxt__(i)
    if q != q2:
        print(f"Mismatch at index {i}:")
        print(f"Original question: {q}")
        print(f"Dataset question: {q2}")
    else:
        train_dset.__setGrade__(i, g)  # Set the grade in the dataset
        break

In [ ]:
train_dset.__getitemInTxt__(0)

In [ ]:
q, a, g = train_dset.__getitemInTxt__(0)
print(f"Question: {q}")
print(f"Answer: {a}")
split_solution_into_subsentences(a)

In [ ]:
start = 0

end = start + 40
# show the answer of the train_examples with igone all new lines
for i in range(start, end):
    example = train_examples[i]
    answer = example["answer"]
    answer_no_newline = answer.replace('\n', ' ')
    print(f"answer{i}:{answer_no_newline}")



In [ ]:
list = [Number, Number, Number, Number, Number, Number, Number, Number, Algebra, Number, Number, Number, Number, Number, Number, Algebra, Number, Number, Number, Number, Number, Number, Number, Number, Number, Number, Number, Number, Algebra, Number, Number, Number, Number, Number, Number, Number, Number, Number, Number]


train_dset.__setGradeByList__([7470,7471,7472], list )

In [ ]:
result = []
for i in range(0, 7473):
    a, b, c = train_dset.__getitemInTxt__(i)
    if c == None:
        result.append(f"{i} No grade")
result

In [ ]:
print(start, end)

In [ ]:
# function to create a list of index by input the start and end index
def create_index_list(start, end):
    return [i for i in range(start, end)]

In [ ]:
import torch
# export train_dset
# torch.save(train_data, 'gsm8k_train_data.pth')

In [ ]:
import torch
train_data = torch.load('gsm8k_train_data.pth')

In [ ]:
train_data.__getSplittedAns__(700)

# Read AQuA

In [ ]:
from aqua_dataset import AQuADataset, get_examples
from transformers import GPT2Tokenizer, GPT2LMHeadModel
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
train_examples = get_examples("dev")
print(train_examples[0])
train_dset = AQuADataset(tokenizer, train_examples[0:1])

In [ ]:
train_dset[0]

# Read MWP

In [ ]:
from mwp_dataset import _load_dataset, MWPDataset
MWP_PATH = "../data/PromptPG/data/tabmwp/"
import argparse
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Create an empty Namespace object
mwp_args = argparse.Namespace()

mwp_args.data_root = MWP_PATH
mwp_args.split = "dev1k"
mwp_args.option_inds = ["A", "B", "C", "D", "E", "F"]
mwp_args.max_length = 100


tokenizer = AutoTokenizer.from_pretrained("gpt2")

train_dset = MWPDataset(MWP_PATH, mwp_args.split, tokenizer, mwp_args)

In [ ]:
train_dset.entries